In [0]:
CREATE OR REPLACE TABLE ifood_case.silver.yellow_trips
USING DELTA
PARTITIONED BY (ano_mes)
AS
SELECT
    CAST(VendorID AS INT)                        AS vendor_id,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    CAST(passenger_count AS INT)                 AS passenger_count,
    CAST(total_amount AS DECIMAL(10,2))          AS total_amount,
    DATE_FORMAT(tpep_pickup_datetime,'yyyy-MM')  AS ano_mes,
    HOUR(tpep_pickup_datetime)                   AS hora_pickup
FROM ifood_case.bronze.yellow_tripdata_raw
WHERE
    tpep_pickup_datetime >= '2023-01-01'
    AND tpep_pickup_datetime <  '2023-06-01'
    AND total_amount    IS NOT NULL
    AND total_amount    >= 0
    AND passenger_count IS NOT NULL
    AND passenger_count >  0
    AND tpep_dropoff_datetime > tpep_pickup_datetime;


num_affected_rows,num_inserted_rows


In [0]:
SELECT
    ano_mes,
    COUNT(*)                    AS corridas,
    ROUND(AVG(total_amount), 2) AS ticket_medio
FROM ifood_case.silver.yellow_trips
GROUP BY ano_mes
ORDER BY ano_mes;


ano_mes,corridas,ticket_medio
2023-01,2917665,27.46
2023-02,2764200,27.37
2023-03,3226999,28.28
2023-04,3109876,28.78
2023-05,3319397,29.45


In [0]:
SELECT
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END)
        AS viola_valor_negativo,
    SUM(CASE WHEN passenger_count <= 0 THEN 1 ELSE 0 END)
        AS viola_passageiros,
    SUM(CASE WHEN tpep_dropoff_datetime <= tpep_pickup_datetime
             THEN 1 ELSE 0 END)
        AS viola_tempo
FROM ifood_case.silver.yellow_trips;


viola_valor_negativo,viola_passageiros,viola_tempo
0,0,0
